In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, PowerTransformer, QuantileTransformer, RobustScaler, StandardScaler

# Busca la raiz del repo aunque el notebook se ejecute desde notebooks/.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from utils import Modelo, basic_comb_train


# Usa una muestra relativamente chica para iterar rapido.
data = pd.read_csv(repo_root / "data" / "training.csv").sample(4000, random_state=42)

# Se deja Weight y EventId porque basic_comb_train usa Weight para AMS
# y saca Weight/EventId antes de entrenar.
X_train = data.drop(columns=["Label"]).replace(-999.0, np.nan)
Y_train = data["Label"]

# Pipelines mas normales.
pipelines = [
    (
        "median_standard",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
    ),
    (
        "median_robust",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]),
    ),
    (
        "median_minmax",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
        ]),
    ),

    # Pipelines un poco mas jugados.
    (
        "knn_standard",
        Pipeline([
            ("imputer", KNNImputer(n_neighbors=5)),
            ("scaler", StandardScaler()),
        ]),
    ),
    (
        "power_standard",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("power", PowerTransformer()),
            ("scaler", StandardScaler()),
        ]),
    ),
    (
        "quantile_robust",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("quantile", QuantileTransformer(output_distribution="normal", n_quantiles=200, random_state=42)),
            ("scaler", RobustScaler()),
        ]),
    ),
    (
        "pca_standard",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=0.95, random_state=42)),
        ]),
    ),
]

models = [
    Modelo(
        "logreg",
        LogisticRegression(max_iter=1500, class_weight="balanced"),
    ),
    Modelo(
        "random_forest",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ),
    ),
    Modelo(
        "gradient_boosting",
        GradientBoostingClassifier(random_state=42),
    ),
]

# Algunas combinaciones se omiten porque no suelen tener mucho sentido
# o agregan costo sin demasiado valor para esta prueba.
dont_train = [
    ("pca_standard", "random_forest"),
    ("pca_standard", "gradient_boosting"),
]

resultados = basic_comb_train(
    X_train=X_train,
    Y_train=Y_train,
    models=models,
    pipelines=pipelines,
    k_fold=3,
    path=repo_root / "modelos_prueba",
    dont_train=dont_train,
)

resumen = pd.DataFrame([
    {
        "pipeline": r["pipeline_name"],
        "modelo": r["model_name"],
        "ams": round(r["mean_ams"], 6),
        "f1": round(r["mean_f1"], 6),
        "accuracy": round(r["mean_accuracy"], 6),
        "precision": round(r["mean_precision"], 6),
        "recall": round(r["mean_recall"], 6),
        "saved_path": str(r["saved_path"]),
    }
    for r in resultados
])

resumen


KeyboardInterrupt: 

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin

class AvgJetPtTransformer(BaseEstimator, TransformerMixin):
    """Agrega la columna avg_jet_pt: PRI_jet_all_pt / PRI_jet_num cuando hay jets, 0 si no."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        jet_num = X["PRI_jet_num"]
        jet_all_pt = X["PRI_jet_all_pt"]
        X["avg_jet_pt"] = np.where(jet_num == 0, 0.0, jet_all_pt / jet_num)
        return X


class LogTransformFeatures(BaseEstimator, TransformerMixin):
    """Aplica log1p(x) a las columnas indicadas, crea columnas log_{nombre} y elimina las originales.

    Preserva NaN para que el imputer los maneje despues.
    """

    def _init_(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        columns = self.columns or []
        for col in columns:
            if col not in X.columns:
                continue
            X[f"log_{col}"] = np.where(X[col].isna(), np.nan, np.log1p(X[col] + 1))
            X = X.drop(columns=[col])
        return X


class DropFeatures(BaseEstimator, TransformerMixin):
    """Elimina las columnas indicadas del DataFrame."""

    def _init_(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        columns = self.columns or []
        to_drop = [col for col in columns if col in X.columns]
        return X.drop(columns=to_drop)


class DropHighlyCorrelatedFeatures(BaseEstimator, TransformerMixin):
    """Elimina features con correlacion de Pearson superior al umbral.

    Aprende durante fit() cuales columnas eliminar y aplica la misma
    eliminacion en transform().
    """

    def _init_(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)
        )
        self.to_drop_ = [
            col for col in upper.columns if (upper[col] > self.threshold).any()
        ]
        return self

    def transform(self, X):
        X = X.copy()
        to_drop = [col for col in self.to_drop_ if col in X.columns]
        return X.drop(columns=to_drop)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.decomposition import PCA
from scipy.stats import uniform
from scipy.stats import randint

# ============================================================

pipe_1 = Pipeline([
    ("avg_jet_pt", AvgJetPtTransformer()),
    ("log_transform", LogTransformFeatures(columns=LOG_FEATURES)),
    ("drop_correlated", DropHighlyCorrelatedFeatures(threshold=0.95)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

pipe_1_param_grid = {
    "preprocessing__drop_correlated__threshold": [0.90, 0.95, 0.98],
    "preprocessing__imputer__strategy": ["median", "most_frequent"],
    "preprocessing__scaler__with_mean": [True, False],
    "preprocessing__scaler__with_std": [True, False]
}

# ============================================================

pipe_2 = Pipeline([
    ("avg_jet_pt", AvgJetPtTransformer()),
    ("log_transform", LogTransformFeatures(columns=LOG_FEATURES)),
    ("drop_correlated", DropHighlyCorrelatedFeatures()),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

pipe_2_param_grid = {
    "preprocessing__drop_correlated__threshold": uniform(0.85, 0.14),
    "preprocessing__imputer__strategy": ["median", "most_frequent"],
    "preprocessing__scaler__with_centering": [True, False],
    "preprocessing__scaler__with_scaling": [True, False],
    "preprocessing__scaler__unit_variance": [True, False],
    "preprocessing__scaler__quantile_range": [
        (10.0, 90.0),
        (20.0, 80.0),
        (25.0, 75.0),
        (30.0, 70.0),
    ],
}

# ============================================================

pipe_3 = Pipeline([
    ("avg_jet_pt", AvgJetPtTransformer()),
    ("log_transform", LogTransformFeatures(columns=LOG_FEATURES)),
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler()),
])

pipe_3_param_grid = {
    "preprocessing__imputer__strategy": ["mean", "median", "most_frequent"],
    "preprocessing__scaler__with_mean": [True, False],
    "preprocessing__scaler__with_std": [True, False],
}

# ============================================================

pipe_4 = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler()),
])

pipe_4_param_grid = {
    "preprocessing__imputer__strategy": ["mean", "median", "most_frequent"],
    "preprocessing__scaler__with_mean": [True, False],
    "preprocessing__scaler__with_std": [True, False],
}

# ============================================================

pipe_5 = Pipeline([
    ("imputer", KNNImputer(random_state=42,n_neighbors=5)),
    ("scaler", StandardScaler()),
])

pipe_5_param_grid = {
    "preprocessing__imputer__n_neighbors": randint(2, 21),
    "preprocessing__imputer__weights": ["uniform", "distance"],
    "preprocessing__scaler__with_mean": [True, False],
    "preprocessing__scaler__with_std": [True, False],
}

# ============================================================

pipe_6 = Pipeline([
    ("imputer", SimpleImputer()),
    ("power", PowerTransformer()),
    ("scaler", StandardScaler()),
])

pipe_6_param_grid = {
    "preprocessing__imputer__strategy": ["mean", "median", "most_frequent"],
    "preprocessing__power__method": ["yeo-johnson", "box-cox"],
    "preprocessing__power__standardize": [True, False],
    "preprocessing__scaler__with_mean": [True, False],
    "preprocessing__scaler__with_std": [True, False],
}

# ============================================================

pipe_7 = Pipeline([
    ("imputer", SimpleImputer()),
    ("quantile", QuantileTransformer(
        random_state=42
    )),
    ("scaler", RobustScaler()),
])

pipe_7_param_grid = {
    "preprocessing__imputer__strategy": ["mean", "median", "most_frequent"],

    "preprocessing__quantile__n_quantiles": randint(50, 301),
    "preprocessing__quantile__output_distribution": ["normal", "uniform"],
    "preprocessing__quantile__subsample": [10_000, 50_000, 100_000, None],

    "preprocessing__scaler__with_centering": [True, False],
    "preprocessing__scaler__with_scaling": [True, False],
    "preprocessing__scaler__unit_variance": [True, False],
    "preprocessing__scaler__quantile_range": [
        (10.0, 90.0),
        (20.0, 80.0),
        (25.0, 75.0),
        (30.0, 70.0),
    ],
}

# ============================================================

pipe_8 = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler()),
    ("pca", PCA(random_state=42)),
])

pipe_8_param_grid = {
    "preprocessing__imputer__strategy": ["mean", "median", "most_frequent"],
    "preprocessing__scaler__with_mean": [True, False],
    "preprocessing__scaler__with_std": [True, False],
    "preprocessing__pca__n_components": uniform(0.80, 0.19),
    "preprocessing__pca__svd_solver": ["full", "auto", "randomized"],
    "preprocessing__pca__whiten": [True, False]
}


NameError: name 'LOG_FEATURES' is not defined

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint, uniform, loguniform
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression


decision_tree = DecisionTreeClassifier(random_state=42)

decision_tree_param_grid = {
    "model__criterion": ["gini", "entropy", "log_loss"],
    "model__max_depth": randint(2, 31),
    "model__min_samples_split": randint(2, 21),
    "model__min_samples_leaf": randint(1, 21),
    "model__max_features": [None, "sqrt", "log2"],
    "model__ccp_alpha": loguniform(1e-5, 1e-1),
}

gradient_boosting = GradientBoostingClassifier(random_state=42)

gradient_boosting_param_grid = {
    "model__n_estimators": randint(50, 401),
    "model__learning_rate": loguniform(1e-3, 3e-1),
    "model__max_depth": randint(2, 8),
    "model__min_samples_split": randint(2, 21),
    "model__min_samples_leaf": randint(1, 21),
    "model__subsample": uniform(0.5, 0.5),   # entre 0.5 y 1.0
    "model__max_features": [None, "sqrt", "log2"],
}

random_forest = RandomForestClassifier(random_state=42, n_jobs=-1)

random_forest_param_grid = {
    "model__n_estimators": randint(100, 801),
    "model__criterion": ["gini", "entropy", "log_loss"],
    "model__max_depth": [None] + list(range(3, 31)),
    "model__min_samples_split": randint(2, 21),
    "model__min_samples_leaf": randint(1, 21),
    "model__max_features": [None, "sqrt", "log2"],
    "model__bootstrap": [True, False],
    "model__ccp_alpha": loguniform(1e-5, 1e-2),
}

xgboost = XGBClassifier(random_state=42)

xgboost_param_grid = {
    "model__n_estimators": randint(50, 501),
    "model__learning_rate": loguniform(1e-3, 3e-1),
    "model__max_depth": randint(2, 11),
    "model__min_child_weight": randint(1, 11),
    "model__subsample": uniform(0.5, 0.5),         # entre 0.5 y 1.0
    "model__colsample_bytree": uniform(0.5, 0.5),  # entre 0.5 y 1.0
    "model__gamma": loguniform(1e-4, 1e1),
    "model__reg_alpha": loguniform(1e-5, 1e1),
    "model__reg_lambda": loguniform(1e-3, 1e2),
}

adaboost = AdaBoostClassifier(random_state=42)

adaboost_param_grid = {
    "model__n_estimators": randint(50, 401),
    "model__learning_rate": loguniform(1e-3, 1.0),
}

logreg = LogisticRegression(random_state=42)

logreg_param_grid = {
    "model__C": loguniform(1e-4, 1e2),
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear", "saga"],
    "model__class_weight": [None, "balanced"],
    "model__max_iter": randint(200, 2001),
}

models_list = [
    (decision_tree, decision_tree_param_grid),
    (gradient_boosting, gradient_boosting_param_grid),
    (random_forest, random_forest_param_grid),
    (xgboost, xgboost_param_grid),
    (adaboost, adaboost_param_grid),
    (logreg, logreg_param_grid),
]

In [ ]:
prep_list = [
    (pipe_1, pipe_1_param_grid),
    (pipe_2, pipe_2_param_grid),
    (pipe_3, pipe_3_param_grid),
    (pipe_4, pipe_4_param_grid),
    (pipe_5, pipe_5_param_grid),
    (pipe_6, pipe_6_param_grid),
    (pipe_7, pipe_7_param_grid),
    (pipe_8, pipe_8_param_grid),
]

models_list = [
    (decision_tree, decision_tree_param_grid),
    (gradient_boosting, gradient_boosting_param_grid),
    (random_forest, random_forest_param_grid),
    (xgboost, xgboost_param_grid),
    (adaboost, adaboost_param_grid),
    (logreg, logreg_param_grid),
]

In [ ]:
full_pipe = Pipeline([
    ("preprocessing", pipe_1),
    ("model", LogisticRegression(max_iter=5000))
])

param_grid = [
    {
        "preprocessing": [pipe_1],
        **pipe_1_param_grid,
        "model": [LogisticRegression(max_iter=5000)],
        "model__C": [0.1, 1, 10],
    },
    {
        "preprocessing": [pipe_2],
        **pipe_2_param_grid,
        "model": [RandomForestClassifier(random_state=42)],
        "model__n_estimators": [100, 300],
        "model__max_depth": [None, 10, 20],
    },
]


for pipe, pipe_params in prep_list:
    for model, model_params in models_list:
        print(f"preprocessing: {pipe}, {pipe_params} model: {model.__class__.__name__}, {model_params}")